# Wetland Loss to Agriculture in a Prairie Pothole Watershed, 2000–2020

**Question:** How much wetland in the Pipestem watershed (HUC8, North Dakota) was converted to cropland between 2000 and 2020?

**Approach in one sentence:** build cloud-free spring Landsat composites for 2000, 2010 and 2020, train a separate
random forest for each snapshot using that period's Cropland Data Layer and the National Wetlands Inventory as labels,
classify each snapshot, and track pixels that went from wetland to cropland.

**Why 2000–2020 and not 1990?** A first version trained one model on 2020 and applied it to 1990, 2000 and 2010.
Sanity checks showed that model did not carry over to older imagery: it mapped implausibly large wetland areas in the
earlier years, including field-shaped "wetlands" during the 1989–1991 drought. Testing variants (with and without a
Landsat 8 → Landsat 7 adjustment, and with indices only) did not fix it (`scripts/diagnose_transfer.py`). The Cropland
Data Layer covers this watershed every year from 1997, so restricting the study to 2000–2020 lets every snapshot have
**its own training labels and its own model**.

Every number reported in this project comes from a cell in this notebook.

## Step 0 — Connect to Earth Engine

Earth Engine runs the heavy processing on Google's servers; this notebook just sends it instructions.
To use it you need a (free, noncommercial) Google Cloud project registered for Earth Engine.
`ee.Initialize(project=...)` tells Earth Engine which project to bill the (free) usage to.

In [1]:
import ee
import geemap

# Your Google Cloud project registered for noncommercial Earth Engine use
PROJECT_ID = "agriculture-wetland-loss"

# Authenticate() only opens a browser the first time; after that it reuses saved credentials.
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized:", ee.String("ok").getInfo())

Earth Engine initialized: ok


## Step 1a — Load the watershed boundary

The USGS Watershed Boundary Dataset (WBD) divides the U.S. into nested hydrologic units.
A **HUC8** is a sub-basin, typically a few thousand km². We look up Pipestem by name and print its
code and area so we can confirm we have the right polygon before doing anything else.

In [2]:
huc8 = ee.FeatureCollection("USGS/WBD/2017/HUC08")
watershed = huc8.filter(ee.Filter.eq("name", "Pipestem"))

# Check what we actually got (there should be exactly one feature, in ND)
info = watershed.first().toDictionary().getInfo()
print("Features matched:", watershed.size().getInfo())
print({k: info[k] for k in sorted(info)})

aoi = watershed.geometry()  # the area of interest used everywhere below
print("Area computed from geometry (km²):", round(aoi.area(maxError=1).divide(1e6).getInfo(), 1))

Features matched: 1
{'areaacres': '685100.43', 'areasqkm': '2772.51', 'gnis_id': '', 'huc8': '10160002', 'loaddate': '20120611075457', 'metasource': '', 'name': 'Pipestem', 'shape_area': '0.329154073822313', 'shape_leng': '4.36477122786687', 'sourcedata': '', 'sourcefeat': '', 'sourceorig': '', 'states': 'ND', 'tnmid': '{C1EC565A-3D79-4C86-9D4E-B1DE84BABDB3}'}


Area computed from geometry (km²): 2764.9


## Step 1b — Cloud masking, scaling, and a common band naming

**Landsat Collection 2 Level-2** gives *surface reflectance*: the fraction of sunlight reflected by the ground,
with atmospheric effects removed. A few housekeeping steps are needed for every image:

1. **Cloud mask.** Each scene has a `QA_PIXEL` band where individual bits flag fill, cloud, dilated cloud,
   cloud shadow and snow. We drop any pixel with one of those flags, plus saturated pixels (`QA_RADSAT`).
2. **Scale factors.** Reflectance is stored as integers; `value × 0.0000275 − 0.2` converts to 0–1 reflectance.
3. **Common band names.** Landsat 5/7 and Landsat 8 number their bands differently
   (e.g. red is `SR_B3` on Landsat 5/7 but `SR_B4` on Landsat 8), so we rename to `blue, green, red, nir, swir1, swir2`.

Note: Landsat 8 sees slightly different wavelengths than Landsat 5/7. We use its surface reflectance as-is
(no cross-sensor adjustment), which keeps the method simple; the possible effect is noted under limitations in the README.

In [3]:
BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
TM_ETM_BANDS = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"]  # Landsat 5 and 7
OLI_BANDS = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]     # Landsat 8


def mask_clouds(img):
    """Keep only clear pixels using the QA_PIXEL bit flags."""
    qa = img.select("QA_PIXEL")
    # bit 0 fill, bit 1 dilated cloud, bit 3 cloud, bit 4 cloud shadow, bit 5 snow
    bad = (1 << 0) | (1 << 1) | (1 << 3) | (1 << 4) | (1 << 5)
    clear = qa.bitwiseAnd(bad).eq(0)
    not_saturated = img.select("QA_RADSAT").eq(0)
    return img.updateMask(clear).updateMask(not_saturated)


def scale(img, band_names):
    """Convert stored integers to surface reflectance and rename to common names."""
    return img.select(band_names, BANDS).multiply(0.0000275).add(-0.2)


def prep_tm_etm(img):
    return scale(mask_clouds(img), TM_ETM_BANDS).copyProperties(img, ["system:time_start"])


def prep_oli(img):
    return scale(mask_clouds(img), OLI_BANDS).copyProperties(img, ["system:time_start"])

## Step 1c — Build the three spring composites

For each snapshot year we take **every clear Landsat pixel from April–June over a 3-year window** and keep the
**median** value per pixel. The median ignores leftover clouds or odd values, and the 3-year window
smooths out a single unusually wet or dry spring.

| Snapshot | Window | Sensors |
|---|---|---|
| 2000 | 1999–2001 | Landsat 5 + 7 |
| 2010 | 2009–2011 | Landsat 5 + 7 (Landsat 7 has striping gaps after 2003; other scenes fill them) |
| 2020 | 2019–2021 | Landsat 8 |

We print how many scenes went in and how many clear observations each pixel had, so we know the composites are well supported.

In [4]:
COLLECTIONS = {
    "L5": ("LANDSAT/LT05/C02/T1_L2", prep_tm_etm),
    "L7": ("LANDSAT/LE07/C02/T1_L2", prep_tm_etm),
    "L8": ("LANDSAT/LC08/C02/T1_L2", prep_oli),
}

SNAPSHOTS = {
    2000: (1999, 2001, ["L5", "L7"]),
    2010: (2009, 2011, ["L5", "L7"]),
    2020: (2019, 2021, ["L8"]),
}


def spring_collection(first_year, last_year, sensors):
    """All cloud-masked April-June scenes over the AOI for the window, merged across sensors."""
    merged = ee.ImageCollection([])
    for s in sensors:
        collection_id, prep = COLLECTIONS[s]
        col = (
            ee.ImageCollection(collection_id)
            .filterBounds(aoi)
            .filterDate(f"{first_year}-01-01", f"{last_year + 1}-01-01")
            .filter(ee.Filter.calendarRange(4, 6, "month"))
            .map(prep)
        )
        merged = merged.merge(col)
    return merged


composites = {}
for year, (y0, y1, sensors) in SNAPSHOTS.items():
    col = spring_collection(y0, y1, sensors)
    composites[year] = col.median().clip(aoi)

    # Clear observations per pixel = how many scenes had a usable value there
    n_clear = col.select("red").count().clip(aoi)
    stats = n_clear.reduceRegion(
        reducer=ee.Reducer.min().combine(ee.Reducer.mean(), sharedInputs=True),
        geometry=aoi, scale=30, maxPixels=1e10,
    ).getInfo()
    print(f"{year} ({y0}-{y1}, {'+'.join(sensors)}): {col.size().getInfo()} scenes | "
          f"clear obs per pixel: min {stats['red_min']}, mean {stats['red_mean']:.1f}")

2000 (1999-2001, L5+L7): 43 scenes | clear obs per pixel: min 1, mean 14.5


2010 (2009-2011, L5+L7): 57 scenes | clear obs per pixel: min 2, mean 23.6


2020 (2019-2021, L8): 45 scenes | clear obs per pixel: min 4, mean 13.2


## Step 1d — Look at the composites

Two views per year:
- **True color** (red, green, blue): what your eye would see.
- **False color** (SWIR1, NIR, red): vegetation shows green, open water and wet soils show dark blue/black, and bare
  fields look pink/brown. This is the easiest way to spot prairie potholes.

Toggle layers in the top-right control to compare years.

In [5]:
m = geemap.Map(basemap="Esri.WorldImagery")
m.centerObject(aoi, 10)

true_color = {"bands": ["red", "green", "blue"], "min": 0, "max": 0.2}
false_color = {"bands": ["swir1", "nir", "red"], "min": 0, "max": 0.35}

for year, img in composites.items():
    m.addLayer(img, true_color, f"{year} true color", shown=False)
    m.addLayer(img, false_color, f"{year} false color", shown=(year in (2000, 2020)))

m.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m

## Step 2a — Add spectral indices

Raw bands are useful, but a few **indices** (simple band math) make the three classes much easier to separate:

| Index | Formula | What it highlights |
|---|---|---|
| **NDVI** (Normalized Difference Vegetation Index) | (NIR − Red) / (NIR + Red) | Green, growing vegetation is high; water is negative; bare spring fields are low. |
| **MNDWI** (Modified Normalized Difference Water Index, Xu 2006) | (Green − SWIR1) / (Green + SWIR1) | Open water is positive; land is mostly negative. Uses SWIR1 because water absorbs it strongly. |
| **Tasseled Cap Wetness** (Crist 1985) | weighted sum of all six bands | Soil and plant moisture. Higher = wetter. Helps with shallow, vegetated wetlands that aren't open water. |

Spring timing helps here: in April–June many crop fields are still bare or newly planted (low NDVI), while
grassland is green and potholes hold snowmelt water.

The wetness coefficients were derived for Landsat 5 TM reflectance. We apply them to all sensors unchanged,
consistent with our decision not to adjust Landsat 8 (noted in the README limitations).

In [6]:
# Crist (1985) Tasseled Cap wetness coefficients for TM reflectance,
# in the order blue, green, red, nir, swir1, swir2
TC_WETNESS = [0.0315, 0.2021, 0.3102, 0.1594, -0.6806, -0.6109]


def add_indices(img):
    """Return the composite with NDVI, MNDWI and Tasseled Cap wetness bands added."""
    ndvi = img.normalizedDifference(["nir", "red"]).rename("ndvi")
    mndwi = img.normalizedDifference(["green", "swir1"]).rename("mndwi")
    # Multiply each band by its coefficient, then add them up across bands
    tcw = (
        img.select(BANDS)
        .multiply(ee.Image.constant(TC_WETNESS))
        .reduce(ee.Reducer.sum())
        .rename("tcw")
    )
    return img.addBands([ndvi, mndwi, tcw])


composites = {year: add_indices(img) for year, img in composites.items()}
print("Bands now in each composite:", composites[2020].bandNames().getInfo())

Bands now in each composite: ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'ndvi', 'mndwi', 'tcw']


## Step 2b — Sanity check: index summaries per snapshot

For each snapshot we print the 10th percentile, median and 90th percentile of each index across the watershed,
plus the share of pixels with **MNDWI > 0**. That last number is only a rough "looks like open water" indicator
(no classification yet), but it shows how wet each window was before we classify anything.

In [7]:
import pandas as pd

rows = []
for year, img in composites.items():
    pct = img.select(["ndvi", "mndwi", "tcw"]).reduceRegion(
        reducer=ee.Reducer.percentile([10, 50, 90]),
        geometry=aoi, scale=30, maxPixels=1e10,
    )
    # Mean of a 0/1 image = fraction of pixels where the condition is true
    water_like = img.select("mndwi").gt(0).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e10,
    )
    row = {"year": year, **pct.getInfo()}
    row["pct_pixels_mndwi_gt_0"] = water_like.get("mndwi").getInfo() * 100
    rows.append(row)

index_summary = pd.DataFrame(rows).set_index("year")
# Order columns as index_p10, index_p50, index_p90
cols = [f"{b}_p{p}" for b in ["ndvi", "mndwi", "tcw"] for p in (10, 50, 90)] + ["pct_pixels_mndwi_gt_0"]
index_summary = index_summary[cols].round(3)
index_summary

,ndvi_p10,ndvi_p50,ndvi_p90,mndwi_p10,mndwi_p50,mndwi_p90,tcw_p10,tcw_p50,tcw_p90,pct_pixels_mndwi_gt_0
year,,,,,,,,,,
2000,0.223,0.363,0.566,-0.551,-0.488,-0.356,-0.287,-0.201,-0.092,4.867
2010,0.184,0.293,0.535,-0.543,-0.504,-0.363,-0.319,-0.214,-0.089,5.059
2020,0.231,0.332,0.551,-0.566,-0.527,-0.434,-0.330,-0.252,-0.123,5.187


## Step 2c — Look at the indices on a map

Blue = wet / water, brown = dry, green = vegetated. Compare 2000 and 2020 for each index with the layer control.

In [8]:
m2 = geemap.Map(basemap="Esri.WorldImagery")
m2.centerObject(aoi, 10)

ndvi_vis = {"min": -0.2, "max": 0.6, "palette": ["8c510a", "f6e8c3", "1a9850"]}
mndwi_vis = {"min": -0.6, "max": 0.4, "palette": ["8c510a", "f5f5f5", "2166ac"]}
tcw_vis = {"min": -0.35, "max": 0.0, "palette": ["8c510a", "f5f5f5", "2166ac"]}

for year in (2000, 2020):
    img = composites[year]
    m2.addLayer(img.select("ndvi"), ndvi_vis, f"{year} NDVI", shown=False)
    m2.addLayer(img.select("tcw"), tcw_vis, f"{year} TC wetness", shown=False)
    m2.addLayer(img.select("mndwi"), mndwi_vis, f"{year} MNDWI", shown=(year == 2020))

m2.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m2